In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Load dataset
train_df = pd.read_csv("train_u6lujuX_CVtuZ9i.csv")
loan_ids_full = train_df['Loan_ID']  # Save for optional test ID usage
train_df.drop('Loan_ID', axis=1, inplace=True)

# Separate features and target
X = train_df.drop("Loan_Status", axis=1)
y = train_df["Loan_Status"].map({'Y': 1, 'N': 0})

# Define categorical and numerical columns
cat_cols = X.select_dtypes(include='object').columns
num_cols = X.select_dtypes(exclude='object').columns

# Preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ]
)

# Three-way split: train, validation, test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.1, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, stratify=y_temp, random_state=42)

# Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    class_weight='balanced'
)

# Full pipeline
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', rf_model)
])

# Train model
model_pipeline.fit(X_train, y_train)

# Save model
joblib.dump(model_pipeline, 'random_forest_loan_model.pkl')

# Load model (for demonstration)
model_pipeline = joblib.load('random_forest_loan_model.pkl')

# Predictions
train_preds = model_pipeline.predict(X_train)
val_preds = model_pipeline.predict(X_val)
test_preds = model_pipeline.predict(X_test)

# Accuracy scores
print("Training Accuracy:", accuracy_score(y_train, train_preds))
print("Validation Accuracy:", accuracy_score(y_val, val_preds))
print("Test Accuracy:", accuracy_score(y_test, test_preds))

# === Validation Set Evaluation ===
print("\nValidation Classification Report:\n", classification_report(y_val, val_preds))

# Confusion Matrix (Validation)
cm_val = confusion_matrix(y_val, val_preds)
plt.figure(figsize=(6, 4))
sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Rejected', 'Approved'],
            yticklabels=['Rejected', 'Approved'])
plt.title('Validation Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

# ROC Curve (Validation)
val_probs = model_pipeline.predict_proba(X_val)[:, 1]
fpr_val, tpr_val, thresholds_val = roc_curve(y_val, val_probs)
roc_auc_val = auc(fpr_val, tpr_val)

plt.figure(figsize=(8, 6))
plt.plot(fpr_val, tpr_val, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc_val:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Validation ROC Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()

# === Test Set Evaluation ===
print("\nTest Set Classification Report:\n", classification_report(y_test, test_preds))

# Confusion Matrix (Test)
cm_test = confusion_matrix(y_test, test_preds)
plt.figure(figsize=(6, 4))
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Rejected', 'Approved'],
            yticklabels=['Rejected', 'Approved'])
plt.title('Test Set Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

# ROC Curve (Test)
test_probs = model_pipeline.predict_proba(X_test)[:, 1]
fpr_test, tpr_test, thresholds_test = roc_curve(y_test, test_probs)
roc_auc_test = auc(fpr_test, tpr_test)

plt.figure(figsize=(8, 6))
plt.plot(fpr_test, tpr_test, color='darkred', lw=2, label=f'ROC Curve (AUC = {roc_auc_test:.2f})')
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Test Set ROC Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()

# === Feature Importance ===
feature_importances = model_pipeline.named_steps['classifier'].feature_importances_
feature_names = model_pipeline.named_steps['preprocessor'].get_feature_names_out()

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importance_df.head(15))
plt.title('Top 15 Feature Importances')
plt.tight_layout()
plt.show()

# === Final Predictions on External Test Set ===
test_df = pd.read_csv("test_Y3wMUE5_7gLdaTN.csv")
loan_ids = test_df['Loan_ID']
X_external_test = test_df.drop('Loan_ID', axis=1)

external_preds = model_pipeline.predict(X_external_test)
external_probs = model_pipeline.predict_proba(X_external_test)[:, 1]

results = pd.DataFrame({
    'Loan_ID': loan_ids,
    'Predicted_Status': ['Y' if x == 1 else 'N' for x in external_preds],
    'Approval_Probability': external_probs,
    'Confidence': ['High' if x > 0.7 else 'Medium' if x > 0.5 else 'Low' for x in external_probs]
})

results = pd.concat([results, test_df.reset_index(drop=True)], axis=1)
results.to_csv('loan_predictions_rf.csv', index=False)

print("\nSample External Test Set Predictions:")
print(results[['Loan_ID', 'Predicted_Status', 'Approval_Probability', 'Confidence']].head(10))

